# Humanistica RDF4J MVP Notebook - modèle actuel

Ce notebook produit une première analyse SPARQL-first directement sur le triple store RDF4J. Il reprend les requêtes métier de l'atelier Humanistica en les adaptant au modèle actuellement déployé.

Sections :

1. Sanity check endpoint
2. Coverage KPI
3. Analyse temporelle
4. Analyse spatiale
5. Démonstration linked data (fédération Wikidata)


In [36]:
!pip install plotly

In [37]:
import plotly.io as pio
pio.renderers.default = "iframe"  # ou "notebook", "jupyterlab", "browser"

In [38]:
import os
import re

import numpy as np
import pandas as pd
import plotly.express as px
import requests

pd.set_option('display.max_rows', 200)
pd.set_option('display.max_colwidth', 120)


In [39]:
SPARQL_ENDPOINT = os.getenv('SPARQL_ENDPOINT')
SPARQL_USER = os.getenv('SPARQL_USER')
SPARQL_PASSWORD = os.getenv('SPARQL_PASSWORD')
SPARQL_TIMEOUT_SECONDS = int(os.getenv('SPARQL_TIMEOUT_SECONDS', '240'))
SPARQL_VERIFY_SSL = os.getenv('SPARQL_VERIFY_SSL', 'true').lower() == 'true'
USER_AGENT = os.getenv('USER_AGENT', 'Humanistica-Analytics/0.2')

missing = [k for k, v in {
    'SPARQL_ENDPOINT': SPARQL_ENDPOINT,
    'SPARQL_USER': SPARQL_USER,
    'SPARQL_PASSWORD': SPARQL_PASSWORD,
}.items() if not v]

if missing:
    raise RuntimeError(f"Variables d'environnement manquantes: {missing}")

print('Endpoint configuré :', SPARQL_ENDPOINT)
print('Vérification SSL :', SPARQL_VERIFY_SSL)
print('Timeout (s) :', SPARQL_TIMEOUT_SECONDS)


Endpoint configuré : https://open-gdb.lod4hss.org/repositories/ANR-DECIDON
Vérification SSL : True
Timeout (s) : 60


In [40]:
def sparql_select(query: str, timeout: int | None = None) -> pd.DataFrame:
    headers = {
        'Accept': 'application/sparql-results+json',
        'User-Agent': USER_AGENT,
    }
    resp = requests.get(
        SPARQL_ENDPOINT,
        params={'query': query},
        headers=headers,
        auth=(SPARQL_USER, SPARQL_PASSWORD),
        timeout=timeout or SPARQL_TIMEOUT_SECONDS,
        verify=SPARQL_VERIFY_SSL,
    )
    if not resp.ok:
        detail = resp.text[:2000].strip()
        raise requests.HTTPError(
            f'{resp.status_code} {resp.reason}: {detail}',
            response=resp,
        )

    payload = resp.json()
    rows = payload.get('results', {}).get('bindings', [])
    return pd.DataFrame([
        {k: v.get('value') for k, v in row.items()}
        for row in rows
    ])


def to_num(series):
    return pd.to_numeric(series, errors='coerce')


## 1. Sanity check endpoint

La sélection du graphe de données se fait par la présence de personnes `crm:E21`. Elle ne dépend donc pas de l'ordre ou de la taille des graphes nommés.


In [41]:
q_graphs = '''
SELECT DISTINCT ?g
WHERE { GRAPH ?g { ?s ?p ?o } }
ORDER BY ?g
'''

df_graphs = sparql_select(q_graphs, timeout=max(SPARQL_TIMEOUT_SECONDS, 240))
df_graphs


,g
0,https://gitlab.huma-num.fr/mpuren/decidon/-/tree/main/graphs/deputes.md
1,https://gitlab.huma-num.fr/mpuren/decidon/-/tree/main/graphs/shacl.md


In [42]:
q_data_graph = '''
PREFIX crm: <http://www.cidoc-crm.org/cidoc-crm/>

SELECT DISTINCT ?g
WHERE { GRAPH ?g { ?person a crm:E21 . } }
LIMIT 2
'''

df_data_graph = sparql_select(q_data_graph, timeout=max(SPARQL_TIMEOUT_SECONDS, 120))
if df_data_graph.empty:
    raise RuntimeError('Aucun graphe de données contenant des personnes crm:E21 trouvé.')
if len(df_data_graph) > 1:
    raise RuntimeError(f'Plusieurs graphes de données candidats trouvés: {df_data_graph.g.tolist()}')

DATA_GRAPH = df_data_graph.iloc[0]['g']
MODEL_GRAPH = next((g for g in df_graphs.get('g', []) if g != DATA_GRAPH), None)

print('Data graph:', DATA_GRAPH)
print('Model graph:', MODEL_GRAPH)


Data graph: https://gitlab.huma-num.fr/mpuren/decidon/-/tree/main/graphs/deputes.md
Model graph: https://gitlab.huma-num.fr/mpuren/decidon/-/tree/main/graphs/shacl.md


In [43]:
q_classes = f'''
SELECT ?class (COUNT(DISTINCT ?s) AS ?n)
WHERE {{ GRAPH <{DATA_GRAPH}> {{ ?s a ?class . }} }}
GROUP BY ?class
ORDER BY DESC(?n)
LIMIT 20
'''

df_classes = sparql_select(q_classes, timeout=max(SPARQL_TIMEOUT_SECONDS, 180))
df_classes['n'] = to_num(df_classes['n'])
df_classes


,n,class
0,16932,http://www.cidoc-crm.org/cidoc-crm/E42
1,12360,https://sdhss.org/ontology/social-life-core/C13
2,9802,http://www.cidoc-crm.org/cidoc-crm/E15
3,9437,https://sdhss.org/ontology/core/C13
4,5868,http://www.cidoc-crm.org/cidoc-crm/E33
5,4901,http://www.cidoc-crm.org/cidoc-crm/E21
6,4901,http://www.cidoc-crm.org/cidoc-crm/E67
7,4901,http://www.cidoc-crm.org/cidoc-crm/E69
8,44,http://www.cidoc-crm.org/cidoc-crm/E74
9,3,http://iflastandards.info/ns/lrm/lrmoo/F2


## 2. Coverage KPI

Dans le modèle actuel, les personnes sont reliées à leur identifiant Wikidata par une attribution `crm:E15`. La valeur `Q...` est portée par `sdh:P9`. Les lieux conservent le chemin `supp:P20` puis `owl:sameAs`.


In [44]:
q_cov = f'''
PREFIX crm: <http://www.cidoc-crm.org/cidoc-crm/>
PREFIX core: <https://sdhss.org/ontology/core/>
PREFIX supp: <https://sdhss.org/ontology/crm-supplement/>
PREFIX sdh: <https://sdhss.org/ontology/shortcuts/>
PREFIX owl: <http://www.w3.org/2002/07/owl#>

SELECT ?metric ?total ?with_wikidata
WHERE {{
  {{
    SELECT
      ('persons_with_wikidata' AS ?metric)
      (COUNT(DISTINCT ?person) AS ?total)
      (COUNT(DISTINCT ?personWith) AS ?with_wikidata)
    WHERE {{
      GRAPH <{DATA_GRAPH}> {{
        ?person a crm:E21 .
        OPTIONAL {{
          ?assignment a crm:E15 ;
                      crm:P140 ?person ;
                      crm:P37 ?eid ;
                      crm:P14 ?actor .
          ?eid sdh:P9 ?qid .
          FILTER(CONTAINS(STR(?actor), '/actor/wikidata'))
          FILTER(REGEX(STR(?qid), '^Q[0-9]+$'))
          BIND(?person AS ?personWith)
        }}
      }}
    }}
  }}
  UNION
  {{
    SELECT
      ('birth_commune_with_wikidata' AS ?metric)
      (COUNT(DISTINCT ?place) AS ?total)
      (COUNT(DISTINCT ?placeWith) AS ?with_wikidata)
    WHERE {{
      GRAPH <{DATA_GRAPH}> {{
        ?place a core:C13 ; supp:P20 ?eid .
        FILTER(CONTAINS(STR(?place), '/BirthCommune/'))
        OPTIONAL {{
          ?eid owl:sameAs ?wd .
          FILTER(STRSTARTS(STR(?wd), 'http://www.wikidata.org/entity/'))
          BIND(?place AS ?placeWith)
        }}
      }}
    }}
  }}
  UNION
  {{
    SELECT
      ('birth_departement_with_wikidata' AS ?metric)
      (COUNT(DISTINCT ?place) AS ?total)
      (COUNT(DISTINCT ?placeWith) AS ?with_wikidata)
    WHERE {{
      GRAPH <{DATA_GRAPH}> {{
        ?place a core:C13 ; supp:P20 ?eid .
        FILTER(CONTAINS(STR(?place), '/BirthDepartement/'))
        OPTIONAL {{
          ?eid owl:sameAs ?wd .
          FILTER(STRSTARTS(STR(?wd), 'http://www.wikidata.org/entity/'))
          BIND(?place AS ?placeWith)
        }}
      }}
    }}
  }}
  UNION
  {{
    SELECT
      ('death_commune_with_wikidata' AS ?metric)
      (COUNT(DISTINCT ?place) AS ?total)
      (COUNT(DISTINCT ?placeWith) AS ?with_wikidata)
    WHERE {{
      GRAPH <{DATA_GRAPH}> {{
        ?place a core:C13 ; supp:P20 ?eid .
        FILTER(CONTAINS(STR(?place), '/DeathCommune/'))
        OPTIONAL {{
          ?eid owl:sameAs ?wd .
          FILTER(STRSTARTS(STR(?wd), 'http://www.wikidata.org/entity/'))
          BIND(?place AS ?placeWith)
        }}
      }}
    }}
  }}
  UNION
  {{
    SELECT
      ('death_departement_with_wikidata' AS ?metric)
      (COUNT(DISTINCT ?place) AS ?total)
      (COUNT(DISTINCT ?placeWith) AS ?with_wikidata)
    WHERE {{
      GRAPH <{DATA_GRAPH}> {{
        ?place a core:C13 ; supp:P20 ?eid .
        FILTER(CONTAINS(STR(?place), '/DeathDepartement/'))
        OPTIONAL {{
          ?eid owl:sameAs ?wd .
          FILTER(STRSTARTS(STR(?wd), 'http://www.wikidata.org/entity/'))
          BIND(?place AS ?placeWith)
        }}
      }}
    }}
  }}
}}
'''

df_cov = sparql_select(q_cov, timeout=max(SPARQL_TIMEOUT_SECONDS, 180))
for column in ['total', 'with_wikidata']:
    df_cov[column] = to_num(df_cov[column])
df_cov['coverage_pct'] = (100 * df_cov['with_wikidata'] / df_cov['total']).round(2)
df_cov = df_cov.sort_values('coverage_pct', ascending=False)
df_cov


,total,with_wikidata,metric,coverage_pct
4,141,131,death_departement_with_wikidata,92.91
2,140,116,birth_departement_with_wikidata,82.86
1,2481,1650,birth_commune_with_wikidata,66.51
3,1997,1327,death_commune_with_wikidata,66.45
0,4901,2594,persons_with_wikidata,52.93


In [45]:
fig_cov = px.bar(
    df_cov,
    x='metric',
    y='coverage_pct',
    text='coverage_pct',
    title='Couverture Wikidata par type d entité',
)
fig_cov.update_traces(texttemplate='%{text:.2f}%', textposition='outside')
fig_cov.update_layout(yaxis_title='Couverture (%)', xaxis_title='Indicateur')
fig_cov.show()


## 3. Analyse temporelle

Les dates sont actuellement des chaînes au format `JJ/MM/AAAA`. La conversion explicite avec pandas est donc conservée.


In [46]:
q_temporal = f'''
PREFIX crm: <http://www.cidoc-crm.org/cidoc-crm/>
PREFIX sdh: <https://sdhss.org/ontology/shortcuts/>

SELECT ?person ?name ?birth_date ?death_date
WHERE {{
  GRAPH <{DATA_GRAPH}> {{
    ?birth a crm:E67 ; crm:P98 ?person ; sdh:P1 ?birth_date .
    ?death a crm:E69 ; crm:P100 ?person ; sdh:P1 ?death_date .
    OPTIONAL {{ ?person sdh:P9 ?name . }}
  }}
}}
'''

df_temp = sparql_select(q_temporal, timeout=max(SPARQL_TIMEOUT_SECONDS, 180))
df_temp['birth_dt'] = pd.to_datetime(df_temp['birth_date'], format='%d/%m/%Y', errors='coerce')
df_temp['death_dt'] = pd.to_datetime(df_temp['death_date'], format='%d/%m/%Y', errors='coerce')
df_temp['age_at_death'] = (df_temp['death_dt'] - df_temp['birth_dt']).dt.days / 365.25
df_temp['birth_decade'] = (df_temp['birth_dt'].dt.year // 10) * 10
df_temp['death_decade'] = (df_temp['death_dt'].dt.year // 10) * 10

n_total = len(df_temp)
valid_birth = df_temp['birth_dt'].notna().sum()
valid_death = df_temp['death_dt'].notna().sum()
valid_age = df_temp['age_at_death'].notna().sum()

print('Lignes extraites:', n_total)
print('Dates de naissance valides:', valid_birth)
print('Dates de décès valides:', valid_death)
print('Âges au décès valides:', valid_age)
print('Lignes écartées de la distribution des âges:', n_total - valid_age)


Lignes extraites: 4901
Dates de naissance valides: 4901
Dates de décès valides: 4900
Âges au décès valides: 4900
Lignes écartées de la distribution des âges: 1


In [47]:
birth_series = (
    df_temp.dropna(subset=['birth_decade'])
    .groupby('birth_decade')
    .size()
    .reset_index(name='n_births')
)

death_series = (
    df_temp.dropna(subset=['death_decade'])
    .groupby('death_decade')
    .size()
    .reset_index(name='n_deaths')
)

fig_birth = px.line(birth_series, x='birth_decade', y='n_births', markers=True, title='Naissances par décennie')
fig_birth.update_layout(xaxis_title='Décennie', yaxis_title='Nombre')
fig_birth.show()

fig_death = px.line(death_series, x='death_decade', y='n_deaths', markers=True, title='Décès par décennie')
fig_death.update_layout(xaxis_title='Décennie', yaxis_title='Nombre')
fig_death.show()

df_age = df_temp[df_temp['age_at_death'].between(10, 110, inclusive='both')].copy()
fig_age = px.histogram(df_age, x='age_at_death', nbins=40, title='Distribution de l âge au décès (10-110 ans)')
fig_age.update_layout(xaxis_title='Âge au décès', yaxis_title='Nombre de personnes')
fig_age.show()

print('Âge médian:', round(df_age['age_at_death'].median(), 2))
print('Âge moyen:', round(df_age['age_at_death'].mean(), 2))


Âge médian: 71.87
Âge moyen: 70.79


## 4. Analyse spatiale

Les événements sont reliés directement par `core:P6` aux nœuds de commune, de département et de lieu composite. Les filtres sur les URI sélectionnent ici uniquement les départements.


In [48]:
q_spatial_top = f'''
PREFIX crm: <http://www.cidoc-crm.org/cidoc-crm/>
PREFIX core: <https://sdhss.org/ontology/core/>
PREFIX sdh: <https://sdhss.org/ontology/shortcuts/>

SELECT ?type ?label (COUNT(DISTINCT ?person) AS ?n)
WHERE {{
  GRAPH <{DATA_GRAPH}> {{
    {{
      ?birth a crm:E67 ; crm:P98 ?person ; core:P6 ?place .
      FILTER(CONTAINS(STR(?place), '/BirthDepartement/'))
      ?place sdh:P9 ?label .
      BIND('BirthDepartement' AS ?type)
    }}
    UNION
    {{
      ?death a crm:E69 ; crm:P100 ?person ; core:P6 ?place .
      FILTER(CONTAINS(STR(?place), '/DeathDepartement/'))
      ?place sdh:P9 ?label .
      BIND('DeathDepartement' AS ?type)
    }}
  }}
}}
GROUP BY ?type ?label
ORDER BY DESC(?n)
'''

df_spatial_top = sparql_select(q_spatial_top, timeout=max(SPARQL_TIMEOUT_SECONDS, 180))
df_spatial_top['n'] = to_num(df_spatial_top['n'])

df_birth_top = df_spatial_top[df_spatial_top['type'] == 'BirthDepartement'].nlargest(20, 'n')
df_death_top = df_spatial_top[df_spatial_top['type'] == 'DeathDepartement'].nlargest(20, 'n')

fig_birth_top = px.bar(df_birth_top.sort_values('n'), x='n', y='label', orientation='h', title='Top 20 départements de naissance')
fig_birth_top.update_layout(xaxis_title='Nombre de personnes', yaxis_title='Département')
fig_birth_top.show()

fig_death_top = px.bar(df_death_top.sort_values('n'), x='n', y='label', orientation='h', title='Top 20 départements de décès')
fig_death_top.update_layout(xaxis_title='Nombre de personnes', yaxis_title='Département')
fig_death_top.show()


In [49]:
q_flows = f'''
PREFIX crm: <http://www.cidoc-crm.org/cidoc-crm/>
PREFIX core: <https://sdhss.org/ontology/core/>
PREFIX sdh: <https://sdhss.org/ontology/shortcuts/>

SELECT ?birth_dep ?death_dep (COUNT(DISTINCT ?person) AS ?n)
WHERE {{
  GRAPH <{DATA_GRAPH}> {{
    ?birth a crm:E67 ; crm:P98 ?person ; core:P6 ?bDepNode .
    ?death a crm:E69 ; crm:P100 ?person ; core:P6 ?dDepNode .
    FILTER(CONTAINS(STR(?bDepNode), '/BirthDepartement/'))
    FILTER(CONTAINS(STR(?dDepNode), '/DeathDepartement/'))
    ?bDepNode sdh:P9 ?birth_dep .
    ?dDepNode sdh:P9 ?death_dep .
  }}
}}
GROUP BY ?birth_dep ?death_dep
ORDER BY DESC(?n)
'''

df_flows = sparql_select(q_flows, timeout=max(SPARQL_TIMEOUT_SECONDS, 180))
df_flows['n'] = to_num(df_flows['n'])

print('Nombre de couples département naissance -> décès:', len(df_flows))
display(df_flows.head(20))

pivot = df_flows.pivot_table(index='birth_dep', columns='death_dep', values='n', fill_value=0)
row_strength = pivot.sum(axis=1).sort_values(ascending=False)
col_strength = pivot.sum(axis=0).sort_values(ascending=False)
heat = pivot.loc[row_strength.head(20).index, col_strength.head(20).index]

fig_heat = px.imshow(
    heat,
    labels={'x': 'Département de décès', 'y': 'Département de naissance', 'color': 'Nombre'},
    title='Matrice des flux naissance -> décès (Top 20 x Top 20)',
    aspect='auto',
)
fig_heat.show()


Nombre de couples département naissance -> décès: 1358


,n,birth_dep,death_dep
0,291,Seine,Seine
1,111,Nord,Nord
2,52,Gironde,Gironde
3,45,Côtes-du-Nord,Côtes-du-Nord
4,44,Nord,Seine
5,41,Finistère,Finistère
6,41,Pas-de-Calais,Pas-de-Calais
7,41,Rhône,Rhône
8,37,Basses-Pyrénées,Basses-Pyrénées
9,37,Isère,Isère


## 5. Démonstration linked data (fédération Wikidata)

Cette section récupère un échantillon de communes déjà alignées avec Wikidata, puis demande leurs coordonnées au service public Wikidata. La fédération peut être désactivée ou limitée par l'administrateur de l'endpoint.


In [50]:
q_map_template = '''
PREFIX core: <https://sdhss.org/ontology/core/>
PREFIX supp: <https://sdhss.org/ontology/crm-supplement/>
PREFIX sdh: <https://sdhss.org/ontology/shortcuts/>
PREFIX owl: <http://www.w3.org/2002/07/owl#>
PREFIX wdt: <http://www.wikidata.org/prop/direct/>

SELECT ?commune ?name ?wd ?coord
WHERE {{
  {{
    SELECT DISTINCT ?commune ?name ?wd
    WHERE {{
      GRAPH <{DATA_GRAPH}> {{
        ?commune a core:C13 ; supp:P20 ?eid ; sdh:P9 ?name .
        FILTER(CONTAINS(STR(?commune), '/BirthCommune/'))
        ?eid owl:sameAs ?wd .
        FILTER(STRSTARTS(STR(?wd), 'http://www.wikidata.org/entity/'))
      }}
    }}
    LIMIT {limit}
  }}
  SERVICE <https://query.wikidata.org/sparql> {{
    ?wd wdt:P625 ?coord .
  }}
}}
'''

attempt_limits = [50, 25, 10]
df_map = pd.DataFrame()
last_error = None

for limit in attempt_limits:
    try:
        q_map = q_map_template.format(limit=limit, DATA_GRAPH=DATA_GRAPH)
        candidate = sparql_select(q_map, timeout=max(SPARQL_TIMEOUT_SECONDS, 180))
        if not candidate.empty and 'coord' in candidate.columns:
            df_map = candidate.copy()
            print(f'Fédération OK avec LIMIT={limit}: {len(df_map)} lignes')
            break
        print(f'La fédération ne retourne aucune coordonnée avec LIMIT={limit}')
    except Exception as exc:
        last_error = exc
        print(f'Échec de la fédération avec LIMIT={limit}: {type(exc).__name__}: {exc}')

if df_map.empty:
    print('La fédération n a retourné aucune coordonnée exploitable.')
    if last_error is not None:
        print('Dernière erreur:', type(last_error).__name__, str(last_error))
else:
    display(df_map.head())


Échec de la fédération avec LIMIT=50: HTTPError: 500 Internal Server Error: <!doctype html><html lang="en"><head><title>HTTP Status 500 – Internal Server Error</title><style type="text/css">body {font-family:Tahoma,Arial,sans-serif;} h1, h2, h3, b {color:white;background-color:#525D76;} h1 {font-size:22px;} h2 {font-size:16px;} h3 {font-size:14px;} p {font-size:12px;} a {color:black;} .line {height:1px;background-color:#525D76;border:none;}</style></head><body><h1>HTTP Status 500 – Internal Server Error</h1><hr class="line" /><p><b>Type</b> Status Report</p><p><b>Message</b> Query evaluation error: org.eclipse.rdf4j.query.QueryEvaluationException: org.eclipse.rdf4j.query.QueryEvaluationException: &lt;html&gt;
&lt;head&gt;&lt;title&gt;502 Bad Gateway&lt;&#47;title&gt;&lt;&#47;head&gt;
&lt;body&gt;
&lt;center&gt;&lt;h1&gt;502 Bad Gateway&lt;&#47;h1&gt;&lt;&#47;center&gt;
&lt;hr&gt;&lt;center&gt;nginx&#47;1.22.1&lt;&#47;center&gt;
&lt;&#47;body&gt;
&lt;&#47;html&gt;</p><p><b>Description</

,wd,commune,name,coord
0,http://www.wikidata.org/entity/Q42168,https://gitlab.huma-num.fr/mpuren/decidon/-/tree/main/graphs/BirthCommune/lieu_naissance_commune%3DClermont-Ferrand,Clermont-Ferrand,Point(3.086944444 45.779722222)
1,http://www.wikidata.org/entity/Q177656,https://gitlab.huma-num.fr/mpuren/decidon/-/tree/main/graphs/BirthCommune/lieu_naissance_commune%3DSpezet,Spezet,Point(-3.716666666 48.1925)
2,http://www.wikidata.org/entity/Q182481,https://gitlab.huma-num.fr/mpuren/decidon/-/tree/main/graphs/BirthCommune/lieu_naissance_commune%3DTourcoing,Tourcoing,Point(3.160277777 50.7225)
3,http://www.wikidata.org/entity/Q209862,https://gitlab.huma-num.fr/mpuren/decidon/-/tree/main/graphs/BirthCommune/lieu_naissance_commune%3DLargenti%C3%A8re,Largentière,Point(4.293333333 44.542777777)
4,http://www.wikidata.org/entity/Q212793,https://gitlab.huma-num.fr/mpuren/decidon/-/tree/main/graphs/BirthCommune/lieu_naissance_commune%3DCharenton-le-pont,Charenton-le-pont,Point(2.411944444 48.821388888)


In [51]:
def parse_wkt_point(wkt: str):
    match = re.match(r'^Point\(([-+0-9.eE]+)\s+([-+0-9.eE]+)\)$', str(wkt))
    if not match:
        return (np.nan, np.nan)
    return float(match.group(1)), float(match.group(2))


if df_map.empty or 'coord' not in df_map.columns:
    print('Aucune coordonnée disponible. Analyse géographique ignorée.')
    df_geo = pd.DataFrame(columns=['name', 'wd', 'lat', 'lon'])
else:
    coords = df_map['coord'].apply(parse_wkt_point)
    df_map['lon'] = coords.apply(lambda point: point[0])
    df_map['lat'] = coords.apply(lambda point: point[1])
    df_geo = df_map.dropna(subset=['lat', 'lon']).copy()
    print('Lignes avec coordonnées valides:', len(df_geo))

if df_geo.empty:
    print('Aucune carte à afficher.')
else:
    fig_geo = px.scatter_geo(
        df_geo,
        lat='lat',
        lon='lon',
        hover_name='name',
        hover_data={'wd': True, 'lat': ':.4f', 'lon': ':.4f'},
        title='Communes de naissance enrichies via Wikidata (échantillon)',
        opacity=0.7,
    )
    fig_geo.update_layout(geo_scope='europe')
    fig_geo.show()


Lignes avec coordonnées valides: 25


## Conclusions rapides

- Le pipeline SPARQL-first fonctionne sur RDF4J avec authentification.
- La couverture des personnes suit désormais les attributions d'identifiants `crm:E15`.
- La couverture des lieux continue d'utiliser `supp:P20` et `owl:sameAs`.
- Les analyses temporelles et spatiales conservent leurs chemins métier.
- La fédération Wikidata reste optionnelle et volontairement limitée pour réduire les risques de timeout.
